# Valider un modèle

| Module | Durée | Prérequis |
|---|---|---|
| M6 – Orthogonalité et moindres carrés | 75 min | M5 |

In [ ]:
# Style graphique Liouaï : couleurs du logo, axes sobres
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

LIOUAI = {"navy": "#243743", "vert": "#28B78D", "vert_texte": "#167A5C", "gris": "#A1A9AE",
          "gris_texte": "#5C6870", "jaune": "#E5C800", "fond": "#F3F5F6"}
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[LIOUAI["vert"], LIOUAI["navy"], LIOUAI["gris"],
                                         LIOUAI["jaune"], LIOUAI["vert_texte"], LIOUAI["gris_texte"]]),
    "axes.edgecolor": LIOUAI["gris"], "axes.labelcolor": LIOUAI["navy"],
    "axes.titlecolor": LIOUAI["navy"], "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": LIOUAI["gris_texte"], "ytick.color": LIOUAI["gris_texte"],
    "text.color": LIOUAI["navy"], "grid.color": "#E1E6E9", "legend.frameon": False,
    "font.family": "sans-serif", "font.sans-serif": ["Roboto", "Helvetica", "Arial", "DejaVu Sans"],
    "figure.dpi": 100,
})
# Dégradé de marque : gris clair -> vert -> navy (et sa version inversée)
if "liouai" not in mpl.colormaps:
    _cmap = LinearSegmentedColormap.from_list("liouai", [LIOUAI["fond"], LIOUAI["vert"], LIOUAI["navy"]])
    mpl.colormaps.register(_cmap)
    mpl.colormaps.register(_cmap.reversed())

## ① Définir la question

La mairie veut prévoir la consommation d'électricité de l'école Jules-Verne, chauffée par des pompes à chaleur. Elle en a besoin pour son budget, mais aussi pour juger, l'an prochain, si les travaux d'isolation ont vraiment fait baisser la facture : il faudra comparer la consommation réelle à celle qu'**aurait eue** l'école sans travaux.

On dispose de deux années scolaires de relevés : pour chaque jour de classe, la température extérieure moyenne et la consommation en kWh. Ces données sont **simulées** à partir d'un modèle réaliste ; elles ont les défauts habituels des vraies.

Un premier stagiaire a ajusté un polynôme de degré 12 : sa courbe passe remarquablement près des points. La responsable de l'énergie se méfie.

**Les questions : comment savoir si un modèle prévoira bien des jours qu'il n'a jamais vus ? Comment savoir s'il lui manque quelque chose ?**

**Activité** : Pourquoi une courbe qui passe très près des points connus ne garantit-elle pas de bonnes prévisions ? Comment pourriez-vous le tester avec les données dont vous disposez ?

Indiquez vos réponses et justifiez-les.

## ② Estimer avant de calculer

In [ ]:
pari_meilleur_degre = 12          # degré du polynôme qui prévoira le mieux l'année suivante (modifiez)

## ③ Abstraire : apprendre sur une partie, juger sur l'autre

**Séparer les données.** On ajuste les modèles sur la première année (données d'**apprentissage**) et on mesure leur erreur sur la seconde (données de **test**), que le modèle n'a pas vues. L'erreur de test estime l'erreur qu'on fera vraiment en prévision.

![Schéma : trois ajustements des mêmes données ; trop simple, l'erreur est élevée partout ; adapté, elle est faible sur les deux ; trop souple, la courbe colle aux données d'apprentissage mais se trompe sur le test](../../assets/schemas/schema-apprentissage-test.svg)

On mesure l'erreur par la **racine de l'erreur quadratique moyenne**, en kWh :

$$
\text{REQM} = \sqrt{\frac{1}{n}\sum_{i=1}^{n} \left(y_i - \hat{y}_i\right)^2 } .
$$

**Analyser les résidus.** Si le modèle a capté tout ce qui est prévisible, les résidus ne doivent plus dépendre de rien : ni de la température, ni du jour de la semaine, ni de la date. Une **forme** dans les résidus signale un ingrédient manquant.

**Des colonnes à volonté.** Un modèle reste « linéaire » pour les moindres carrés tant qu'il est une combinaison linéaire de colonnes connues, même si ces colonnes sont des fonctions compliquées des données : $T^2$, $\max(0,\ 15 - T)$, ou une colonne qui vaut 1 le mercredi et 0 sinon.

**Activité** : Pour une pompe à chaleur, à quoi ressemble, selon vous, la consommation en fonction de la température extérieure ? Faites un croquis.

Indiquez vos réponses et justifiez-les.

## ④ Calculer

### Les données

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def simuler_annee(debut, rng):
    jours = pd.bdate_range(debut, periods=260)
    jours = jours[~((jours.month == 7) | (jours.month == 8))]               # pas d'école l'été
    jours = jours[~jours.isocalendar().week.isin([44, 52, 1, 8, 16]).to_numpy()]       # quelques vacances
    jour_annee = np.asarray(jours.dayofyear)
    T = 11 - 8 * np.cos(2 * np.pi * (jour_annee - 20) / 365) + rng.normal(0, 3, len(jours))
    mercredi = np.asarray(jours.dayofweek == 2)
    conso = 420 + 38 * np.maximum(0, 15 - T) - 90 * mercredi + rng.normal(0, 25, len(jours))
    return pd.DataFrame({"date": jours, "temperature": T.round(1), "jour": jours.day_name(),
                         "conso_kwh": conso.round(0)})

rng = np.random.default_rng(64)
ecole = pd.concat([simuler_annee("2024-09-02", rng).assign(annee="2024-2025"),
                   simuler_annee("2025-09-01", rng).assign(annee="2025-2026")], ignore_index=True)
apprentissage = ecole[ecole["annee"] == "2024-2025"]
test = ecole[ecole["annee"] == "2025-2026"]
print(f"{len(apprentissage)} jours d'apprentissage, {len(test)} jours de test")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(apprentissage["temperature"], apprentissage["conso_kwh"], "o", ms=3, color=LIOUAI["navy"], label="apprentissage (2024-2025)")
ax.plot(test["temperature"], test["conso_kwh"], "s", ms=3, color=LIOUAI["jaune"], label="test (2025-2026)")
ax.set_xlabel("température extérieure moyenne (°C)"); ax.set_ylabel("consommation (kWh)"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

### Un outil commun : ajuster et mesurer

Toutes les colonnes passent par la même fonction. On remet la température à l'échelle (module M5) pour que les polynômes de haut degré restent calculables.

In [ ]:
def reqm(y, y_prevu):
    return np.sqrt(np.mean((y - y_prevu) ** 2))

def evaluer(fabriquer_colonnes, nom=""):
    A_app, A_test = fabriquer_colonnes(apprentissage), fabriquer_colonnes(test)
    y_app, y_test = apprentissage["conso_kwh"].to_numpy(), test["conso_kwh"].to_numpy()
    coef, *_ = np.linalg.lstsq(A_app, y_app, rcond=None)
    e_app, e_test = reqm(y_app, A_app @ coef), reqm(y_test, A_test @ coef)
    if nom:
        print(f"{nom:40s} REQM apprentissage {e_app:6.1f}   test {e_test:6.1f} kWh")
    return coef, e_app, e_test

def polynome(degre):
    return lambda df: np.vander((df["temperature"].to_numpy() - 10) / 10, degre + 1, increasing=True)

coef_1, *_ = evaluer(polynome(1), "droite")

### Les résidus de la droite

In [ ]:
A = polynome(1)(apprentissage)
residus_droite = apprentissage["conso_kwh"].to_numpy() - A @ coef_1
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.axhline(0, color=LIOUAI["gris"])
ax.plot(apprentissage["temperature"], residus_droite, "o", ms=3, color=LIOUAI["navy"])
ax.set_xlabel("température (°C)"); ax.set_ylabel("résidu (kWh)"); ax.grid(alpha=0.3)
ax.set_title("Résidus de la droite : ils ont une forme")
plt.tight_layout(); plt.show()

### Monter en degré : apprentissage contre test

In [ ]:
degres = range(1, 13)
erreurs = np.array([evaluer(polynome(d))[1:] for d in degres])

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.plot(degres, erreurs[:, 0], "o-", color=LIOUAI["navy"], label="apprentissage")
ax.plot(degres, erreurs[:, 1], "s-", color=LIOUAI["jaune"], label="test")
ax.set_xlabel("degré du polynôme"); ax.set_ylabel("REQM (kWh)"); ax.legend(); ax.grid(alpha=0.3)
ax.set_title("L'erreur d'apprentissage baisse toujours ; l'erreur de test, non")
plt.tight_layout(); plt.show()
print("Meilleur degré sur le test :", degres[int(np.argmin(erreurs[:, 1]))])

### Un modèle inspiré par la physique

Une pompe à chaleur ne chauffe que lorsqu'il fait froid. En dessous d'une température de consigne, elle consomme d'autant plus qu'il fait froid ; au-dessus, presque rien. Les chauffagistes utilisent pour cela les **degrés-jours** : $\max(0,\ 15 - T)$. On ajoute cette colonne à une colonne constante.

In [ ]:
def degres_jours(seuil=15):
    return lambda df: np.column_stack([np.ones(len(df)), np.maximum(0, seuil - df["temperature"].to_numpy())])

coef_dj, *_ = evaluer(degres_jours(), "constante + degrés-jours (2 paramètres)")
evaluer(polynome(4), "polynôme de degré 4 (5 paramètres)");

### Les résidus contre le jour de la semaine

In [ ]:
A = degres_jours()(apprentissage)
residus_dj = apprentissage["conso_kwh"].to_numpy() - A @ coef_dj
ordre = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
noms = ["lundi", "mardi", "mercredi", "jeudi", "vendredi"]
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.boxplot([residus_dj[apprentissage["jour"].to_numpy() == j] for j in ordre], medianprops=dict(color=LIOUAI["vert"], lw=2))
ax.set_xticks(range(1, 6), noms)
ax.axhline(0, color=LIOUAI["gris"]); ax.set_ylabel("résidu (kWh)"); ax.grid(alpha=0.3)
ax.set_title("Résidus du modèle « degrés-jours » selon le jour")
plt.tight_layout(); plt.show()

**Activité** : Que remarquez-vous ? Quelle explication proposez-vous, connaissant le rythme d'une école primaire ?

Indiquez vos réponses et justifiez-les.

In [ ]:
def complet(seuil=15):
    def colonnes(df):
        return np.column_stack([np.ones(len(df)),
                                np.maximum(0, seuil - df["temperature"].to_numpy()),
                                (df["jour"] == "Wednesday").to_numpy().astype(float)])
    return colonnes

coef_final, e_app, e_test = evaluer(complet(), "constante + degrés-jours + mercredi")
print(f"\nConsommation de base : {coef_final[0]:.0f} kWh ; "
      f"+{coef_final[1]:.1f} kWh par degré sous 15 °C ; mercredi : {coef_final[2]:+.0f} kWh")

y_test = test["conso_kwh"].to_numpy()
residus_test = y_test - complet()(test) @ coef_final
fig, axes = plt.subplots(1, 2, figsize=(13, 3.6))
axes[0].plot(test["date"], residus_test, "o", ms=3, color=LIOUAI["navy"]); axes[0].axhline(0, color=LIOUAI["gris"])
axes[0].set_title("Résidus de test au fil de l'année"); axes[0].set_ylabel("kWh"); axes[0].tick_params(axis="x", rotation=30)
axes[1].hist(residus_test, bins=20, color=LIOUAI["vert"]); axes[1].set_title("Répartition des résidus de test")
for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## ⑤ Interpréter

In [ ]:
print(f"Votre pari : degré {pari_meilleur_degre}")
print(f"Meilleur polynôme sur le test : degré {degres[int(np.argmin(erreurs[:, 1]))]}, REQM {erreurs[:, 1].min():.1f} kWh")
print(f"Modèle final à 3 paramètres   : REQM {e_test:.1f} kWh")

**Activité** :

1. Rédigez pour la mairie le modèle retenu, en une phrase par coefficient, avec ses unités.
2. Pourquoi le polynôme de degré 12 du stagiaire est-il un mauvais choix, alors qu'il colle mieux aux données d'apprentissage ?
3. Avec ce modèle, comment la mairie pourra-t-elle juger, l'an prochain, de l'effet des travaux d'isolation ? Quelle différence de consommation pourrait-on attribuer au hasard ?
4. Les résidus de test ont-ils encore une forme ? Que pourrait-on encore chercher ?

Indiquez vos réponses et justifiez-les.

## ⑥ Critiquer et généraliser

- **Le test a-t-il été bien utilisé ?** On a regardé l'erreur de test pour choisir le degré, puis le modèle. À force de choisir d'après le test, il influence le modèle : il n'est plus tout à fait « jamais vu ». La pratique recommandée réserve une troisième partie des données, ou utilise la **validation croisée**.
- **Le seuil de 15 °C** a été choisi « par la physique ». Pourrait-on l'ajuster aux données ? Le modèle serait-il encore linéaire en ses paramètres ?
- **Généraliser.** Écrivez une fonction `validation_croisee(fabriquer_colonnes, donnees, k)` : elle découpe les données en $k$ paquets, ajuste le modèle sur $k - 1$ paquets, mesure la REQM sur le paquet restant, recommence pour chaque paquet, et renvoie la moyenne. Utilisez-la sur les deux années réunies pour choisir le seuil des degrés-jours entre 10 °C et 20 °C.

In [ ]:
def validation_croisee(fabriquer_colonnes, donnees, k=5):
    # À compléter
    ...

## Corrigé

*À n'ouvrir qu'après avoir cherché.*

In [ ]:
def validation_croisee(fabriquer_colonnes, donnees, k=5, graine=0):
    melange = np.random.default_rng(graine).permutation(len(donnees))
    paquets = np.array_split(melange, k)
    erreurs = []
    for paquet in paquets:
        a_garder = np.setdiff1d(melange, paquet)
        app, val = donnees.iloc[a_garder], donnees.iloc[paquet]
        coef, *_ = np.linalg.lstsq(fabriquer_colonnes(app), app["conso_kwh"].to_numpy(), rcond=None)
        erreurs.append(reqm(val["conso_kwh"].to_numpy(), fabriquer_colonnes(val) @ coef))
    return np.mean(erreurs)

seuils = np.arange(10, 20.5, 0.5)
scores = [validation_croisee(complet(s), ecole) for s in seuils]
fig, ax = plt.subplots(figsize=(7, 3.4))
ax.plot(seuils, scores, "o-", color=LIOUAI["navy"])
ax.set_xlabel("seuil des degrés-jours (°C)"); ax.set_ylabel("REQM en validation croisée (kWh)"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()
print(f"Meilleur seuil : {seuils[int(np.argmin(scores))]} °C")

## Bilan — auto-évaluation

Pour chaque compétence travaillée dans ce notebook, choisissez un niveau : **0** non observé · **1** guidé · **2** autonome · **3** transfert.
La cellule enregistre votre bilan dans `progression.json`, à la racine du site.

In [ ]:
import json, datetime, pathlib

bilan = {
    "Corroborer ses résultats": 1,
    "Identifier erreurs systématiques et aléatoires": 1,
    "Sentir la fiabilité d'un modèle": 1,
    "Combiner des outils pour produire le résultat": 1,
    "Identifier, énoncer et justifier les hypothèses": 1,
}   # modifiez les niveaux

ici = pathlib.Path.cwd()
racine = next((p for p in [ici, *ici.parents] if (p / "myst.yml").exists()), ici)
fichier = racine / "progression.json"
historique = json.loads(fichier.read_text(encoding="utf-8")) if fichier.exists() else []
historique.append({"notebook": "6.4-valider-un-modele", "date": datetime.date.today().isoformat(), "niveaux": bilan})
fichier.write_text(json.dumps(historique, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Bilan enregistré ({len(historique)} entrée(s) dans {fichier.name}).")